## Understanding the differnce between the Embedding Layer and Linear Layers..

* Embedding layers in PyTorch accomplish the same as linear layer that perform matrix mulplication, the reason we use embedding layer is computational efficiency.

* We will take a look at relationship step by step using code examples in PyTorch


In [1]:
import torch

print("PyTorch version", torch.__version__)

PyTorch version 2.9.0+cu126


#### Usig nn.Embedding


In [2]:
# Suppose we've the following 3 training examples  which we reprsent token IDs in a LLM context

idx = torch.tensor([2, 3, 1])

# The f the highest token ID is 3, then we want 4 rows, for the possible token IDs 0, 1, 2, 3
num_idx = max(idx) + 1

# the desired embedding dimension is a hyperparameter
out_dim = 5


* Implementation of a simple `Embedding layer`


In [3]:
# we use the random seed for reproducibility
torch.manual_seed(123)

embedding = torch.nn.Embedding(num_idx, out_dim)

we can optionally take a look at the embedding weights


In [4]:
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953]], requires_grad=True)

* we can then use the embedding layers to obtain the vector representation of a training examle with ID 1:

In [5]:
embedding(torch.tensor([1]))
# Prints the first row of the tensor ...

tensor([[ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

* similarly we can use embedding to obtaion the vector representation of a training example with ID 2:


In [6]:
embedding(torch.tensor([2]))

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315]],
       grad_fn=<EmbeddingBackward0>)

* Now let convert all the training examples we've defined previously


In [7]:
idx = torch.tensor([2, 3, 1])
embedding(idx)
# the position of the raws changed ......2 becomes first, 3 second etc.

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

## Using nn.Linear
* Now we'll  demonstrate that the embedding acomplishes exactly the same as `nn.Linear` layer on the a one-hot encoded representation in PyTorch.

* First, let's convert the token IDs into a one-hot representation.

In [8]:
onehot = torch.nn.functional.one_hot(idx)
onehot

tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])

* Next we initialize a `Linear` layer  which carry out a matrix multiplication XW(T)


In [9]:
torch.manual_seed(123)
linear = torch.nn.Linear(num_idx, out_dim, bias=False)
linear.weight

Parameter containing:
tensor([[-0.2039,  0.0166, -0.2483,  0.1886],
        [-0.4260,  0.3665, -0.3634, -0.3975],
        [-0.3159,  0.2264, -0.1847,  0.1871],
        [-0.4244, -0.3034, -0.1836, -0.0983],
        [-0.3814,  0.3274, -0.1179,  0.1605]], requires_grad=True)

* Note that the linear layer in PyTorch is also initialized with small random weights; to directly compare it to the `Embedding` layer above, we have to use the same small random weights, which is why we reassign them here:



In [10]:
linear.weight = torch.nn.Parameter(embedding.weight.T)


* Now we can use the linear layer on the one-hot encoded representation of the inputs

In [11]:
linear(onehot.float())

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)

* As we can see, this is exactly the same as what we got when we used the embedding layer:


In [13]:
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

* Since all but one index in each one-hot encoded row are 0 (by design), this matrix multiplication is essentially the same as a look-up of the one-hot elements
* This use of the matrix multiplication on one-hot encodings is equivalent to the embedding layer look-up but can be inefficient if we work with large embedding matrices, because there are a lot of wasteful multiplications by zero

### NOTE:
#### Under the hood what happens here is the matrix multplication, the column matrix say [2 3 1] ^T is being multipied by the weight matirx.